In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

100%|██████████| 7.13M/7.13M [00:00<00:00, 115MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/2


Converting BBB to MBM

In [ ]:
df = pd.read_csv(path + "/ganjoo_ipl_bbb.csv")

In [ ]:
df.drop(columns=['wagonX', 'wagonY', 'wagonZone', 'line','length', 'shot', 'control', 'predscore', 'wprob'], inplace=True)

In [ ]:
df.columns

Index(['p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl',
       'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal',
       'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs',
       'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
       'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
       'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'date',
       'year', 'ground', 'country', 'winner', 'toss', 'competition',
       'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced',
       'bowlruns', 'bat_out'],
      dtype='object')

In [ ]:
print(df.sample().iloc[0])

p_match                            1136597
inns                                     2
bat                             Chris Lynn
p_bat                               326637
team_bat             Kolkata Knight Riders
bowl                  Mitchell McClenaghan
p_bowl                              319439
team_bowl                   Mumbai Indians
ball                                     4
ball_id                               0.04
outcome                             no run
score                                    0
out                                  False
dismissal                              NaN
p_out                               326637
over                                     1
noball                                   0
wide                                     0
byes                                     0
legbyes                                  0
cur_bat_runs                             1
cur_bat_bf                               3
cur_bowl_ovr                           0.4
cur_bowl_wk

In [29]:
import pandas as pd

def process_batting_scorecard_from_kaggle(df):
    """
    Process a ball-by-ball DataFrame from the Kaggle dataset into a batting scorecard format.
    This version processes the data match by match so that each match's date (and venue) is correctly assigned.

    Parameters:
      df (DataFrame): The ball-by-ball match data, assumed to be pre-filtered.

    Returns:
      DataFrame: The aggregated batting scorecard with:
               - bat_team: Batting team (from team_bat column)
               - bowl_team: Bowling team (from team_bowl column)
               - bowl_id: For dismissed batsmen, the bowler's ID taken from p_out.
               - venue: The match venue (from ground column)
               - date: The match date specific to each match
    """
    if df.empty:
        return pd.DataFrame()

    all_results = []
    # Process data match by match.
    for match_id in df['p_match'].unique():
        match_df = df[df['p_match'] == match_id]
        # Get match-specific values.
        match_date = (match_df['date'].iloc[0]
                      if 'date' in match_df.columns and not pd.isna(match_df['date'].iloc[0])
                      else "")
        venue = (match_df['ground'].iloc[0]
                 if 'ground' in match_df.columns and not pd.isna(match_df['ground'].iloc[0])
                 else "")

        # Initialize a dictionary to accumulate batting statistics for this match.
        # Use key: (match_id, batting_team, striker, innings) to handle multiple innings.
        batting_stats = {}

        for _, ball in match_df.iterrows():
            striker = pd.to_numeric(ball['p_bat'], errors='coerce')
            batsman_name = ball['bat']          # Batsman's name
            bowler_name = ball['bowl']            # Bowler's name for the current ball
            batting_team = ball['team_bat']       # Batting team name
            bowling_team = ball['team_bowl']      # Bowling team name
            innings = ball['inns']                # Innings

            key = (match_id, batting_team, striker, innings)
            if key not in batting_stats:
                batting_stats[key] = {
                    'batsman_name': batsman_name,
                    'bowler_name': "",    # To be set on dismissal.
                    'runs': 0,
                    'balls': 0,
                    'fours': 0,
                    'sixes': 0,
                    'isOut': False,
                    'wicketType': '',
                    'p_bat_id': striker,
                    'p_bowl_id': None,    # Will be set when a wicket falls.
                    'bat_team': batting_team,
                    'bowl_team': bowling_team
                }
            stats = batting_stats[key]

            # Update runs and balls.
            runs = ball['batruns']
            stats['balls'] += 1
            stats['runs'] += runs

            # Count boundaries.
            if runs == 4:
                stats['fours'] += 1
            elif runs == 6:
                stats['sixes'] += 1
            elif runs == 5:
                stats['fours'] += 1  # Count a 5 as a four (with an extra wide).

            # Check if dismissal occurred.
            out_status = str(ball['out']).strip().upper()
            dismissed_player = str(ball['p_out']).strip()
            if out_status == 'TRUE' and dismissed_player == str(striker):
                stats['isOut'] = True
                stats['wicketType'] = str(ball['dismissal']).strip()
                stats['bowler_name'] = bowler_name  # Set bowler on dismissal.
                # Use bowler's ID from p_out.
                stats['p_bowl_id'] = pd.to_numeric(ball['p_out'], errors='coerce')

            batting_stats[key] = stats

        # Append match results.
        for (m_id, team, striker, innings), stats in batting_stats.items():
            strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
            all_results.append({
                'MatchID': m_id,
                'date': match_date,
                'venue': venue,
                'innings': innings,
                'batsman': striker,
                'batsman_name': stats['batsman_name'],
                'bowler_name': stats['bowler_name'],
                'bowl_id': stats['p_bowl_id'],
                'runs': stats['runs'],
                'balls': stats['balls'],
                'fours': stats['fours'],
                'sixes': stats['sixes'],
                'strikeRate': round(strike_rate, 2),
                'isOut': stats['isOut'],
                'wicketType': stats['wicketType'] if stats['isOut'] else 'not out',
                'bat_team': stats['bat_team'],
                'bowl_team': stats['bowl_team']
            })

    return pd.DataFrame(all_results)


In [30]:
batting_card=process_batting_scorecard_from_kaggle(df)

In [31]:
batting_card.columns

Index(['MatchID', 'date', 'venue', 'innings', 'batsman', 'batsman_name',
       'bowler_name', 'bowl_id', 'runs', 'balls', 'fours', 'sixes',
       'strikeRate', 'isOut', 'wicketType', 'bat_team', 'bowl_team'],
      dtype='object')

In [32]:
batting_card.sort_values(by=['date', 'MatchID', 'innings'], inplace=True)

In [33]:
batting_card = batting_card.rename(columns={'MatchID': 'match_id','innings':'inns','batsman':'batter_id','batsman_name':'batter','bowler_name':'bowler','bowl_id':'bowler_id','strikeRate':'sr'})

In [34]:
batting_card = batting_card.reindex(columns=['date', 'match_id', 'bat_team', 'bowl_team', 'inns','batter_id','batter','runs','balls','fours','sixes','sr','isOut','wicketType','bowler_id', 'bowler','venue'])

In [35]:
batting_card['venue'] = batting_card['venue'].str.split(',').str.get(0)


In [36]:
batting_card.sample()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
7843,2023-04-22,1359504,Lucknow Super Giants,Gujarat Titans,2,1151270,Ayush Badoni,8,6,0,0,133.33,False,not out,NaN,,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...


In [37]:
batting_card[batting_card['match_id'] == 829707]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
6640,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,28081,MS Dhoni,30,27,1,2,111.11,True,caught,28081.0,Nathan Coulter-Nile,MA Chidambaram Stadium
6641,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,44828,Faf du Plessis,32,25,3,0,128.00,True,caught,44828.0,Jean-Paul Duminy,MA Chidambaram Stadium
6642,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,53118,Dwayne Smith,34,31,6,0,109.68,True,caught,53118.0,Imran Tahir,MA Chidambaram Stadium
6643,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,234675,Ravindra Jadeja,17,19,1,1,89.47,True,stumped,234675.0,Amit Mishra,MA Chidambaram Stadium
6644,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,26421,Ravichandran Ashwin,12,8,1,0,150.00,False,not out,NaN,,MA Chidambaram Stadium
6645,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,37737,Brendon McCullum,4,2,1,0,200.00,True,caught,37737.0,Nathan Coulter-Nile,MA Chidambaram Stadium
6646,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,33335,Suresh Raina,4,7,0,0,57.14,True,bowled,33335.0,Nathan Coulter-Nile,MA Chidambaram Stadium
6647,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,537119,Mohit Sharma,2,1,0,0,200.00,False,not out,NaN,,MA Chidambaram Stadium
6654,2015-04-09,829707,Chennai Super Kings,Delhi Capitals,1,51439,Dwayne Bravo,1,3,0,0,33.33,True,leg before wicket,51439.0,Domnic Muthuswami,MA Chidambaram Stadium
6648,2015-04-09,829707,Delhi Capitals,Chennai Super Kings,2,46248,Albie Morkel,73,59,8,1,123.73,False,not out,NaN,,MA Chidambaram Stadium


In [38]:
batting_card.to_csv('ganjoo_ipl_mbm_bat.csv', index=False)

In [39]:
batting_card["bat_team"].unique()


array(['Mumbai Indians', 'Kolkata Knight Riders', 'Chennai Super Kings',
       'Delhi Capitals', 'Rajasthan Royals', 'Punjab Kings',
       'Sunrisers Hyderabad', 'Royal Challengers Bangalore',
       'Rising Pune Supergiant', 'Gujarat Lions', 'Lucknow Super Giants',
       'Gujarat Titans', 'Royal Challengers Bengaluru'], dtype=object)